# PDF Ingestion Pipeline - Quick Start

This notebook demonstrates the RAG ingestion pipeline:
1. Parse PDFs with Docling
2. Chunk the parsed content
3. Inspect the results

**Prerequisites**: Run from the dev container with all dependencies installed.

## 1. Setup

In [1]:
import sys
from pathlib import Path

# Ensure the project root is in the path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from rag.ingestion.pdf_parser import parse_pdf
from rag.ingestion.chunker import chunk_elements

DOCS_DIR = project_root / "pdf"
print(f"Docs directory: {DOCS_DIR}")
print(f"PDF files found: {list(DOCS_DIR.glob('*.pdf'))}")

Docs directory: /workspace/docs
PDF files found: [PosixPath('/workspace/docs/10Q-Q1-2026-as-filed.pdf'), PosixPath('/workspace/docs/GOOG-10-Q-Q1-2026.pdf'), PosixPath('/workspace/docs/10Q-Q2-2026-as-filed.pdf'), PosixPath('/workspace/docs/form-10-q.pdf')]


## 2. List Available PDFs

In [2]:
pdf_files = sorted(DOCS_DIR.glob("*.pdf"))

for i, f in enumerate(pdf_files, 1):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"{i}. {f.name} ({size_mb:.1f} MB)")

1. 10Q-Q1-2026-as-filed.pdf (0.8 MB)
2. 10Q-Q2-2026-as-filed.pdf (0.9 MB)
3. GOOG-10-Q-Q1-2026.pdf (0.6 MB)
4. form-10-q.pdf (0.8 MB)


## 3. Parse a Single PDF

In [3]:
# Parse the first PDF
pdf_path = pdf_files[0]
print(f"Parsing: {pdf_path.name}")

elements = parse_pdf(pdf_path)
print(f"\nTotal elements extracted: {len(elements)}")

Parsing: 10Q-Q1-2026-as-filed.pdf


onnxruntime cpuid_info warning: Unknown CPU vendor. cpuinfo_vendor value: 0
[INFO] 2026-05-08 17:49:30,510 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-08 17:49:30,531 [RapidOCR] download_file.py:60: File exists and is valid: /opt/python-3.11-dev/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-08 17:49:30,533 [RapidOCR] main.py:57: Using /opt/python-3.11-dev/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-08 17:49:30,595 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-08 17:49:30,597 [RapidOCR] download_file.py:60: File exists and is valid: /opt/python-3.11-dev/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-08 17:49:30,597 [RapidOCR] main.py:57: Using /opt/python-3.11-dev/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-08 17:49:30,622 [RapidOCR] base.py:22: Using en


Total elements extracted: 333


In [4]:
# Summary by type
from collections import Counter

type_counts = Counter(e.type for e in elements)
print("Elements by type:")
for t, count in type_counts.most_common():
    print(f"  {t}: {count}")

Elements by type:
  text: 207
  heading: 102
  table: 24


In [5]:
# Preview first 10 elements
for i, elem in enumerate(elements[:10], 1):
    preview = elem.content[:100].replace("\n", " ")
    print(f"{i:2d}. [{elem.type:7s}] p.{elem.page} | {preview}...")

 1. [heading] p.1 | UNITED STATES SECURITIES AND EXCHANGE COMMISSION...
 2. [text   ] p.1 | Washington, D.C. 20549...
 3. [heading] p.1 | FORM 10-Q...
 4. [text   ] p.1 | (Mark One)...
 5. [text   ] p.1 | ☒ QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934...
 6. [text   ] p.1 | For the quarterly period ended December 27, 2025...
 7. [text   ] p.1 | or...
 8. [text   ] p.1 | ☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934...
 9. [text   ] p.1 | For the transition period from              to             ....
10. [text   ] p.1 | Commission File Number:...


## 4. Inspect Tables

In [6]:
tables = [e for e in elements if e.type == "table"]
print(f"Tables found: {len(tables)}")

if tables:
    print(f"\n--- Table 1 (page {tables[0].page}) ---")
    print(tables[0].content[:500])

Tables found: 24

--- Table 1 (page 4) ---
|                                              | Three Months Ended   | Three Months Ended   |
|----------------------------------------------|----------------------|----------------------|
|                                              | December 27, 2025    | December 28, 2024    |
| Net sales:                                   |                      |                      |
| Products                                     | $ 113,743            | $ 97,960             |
| Services               


## 5. Chunk the Parsed Content

In [7]:
# Try all three chunking methods
for method in ["recursive", "semantic", "by_title"]:
    chunks = chunk_elements(
        elements,
        method=method,
        chunk_size=1000,
        chunk_overlap=200,
        keep_tables_intact=True,
    )
    table_chunks = [c for c in chunks if c.type == "table"]
    print(f"{method:10s}: {len(chunks):4d} chunks ({len(table_chunks)} tables intact)")

recursive :  343 chunks (24 tables intact)
semantic  :   98 chunks (24 tables intact)
by_title  :  336 chunks (24 tables intact)


In [8]:
# Inspect chunks from recursive method
chunks = chunk_elements(
    elements,
    method="recursive",
    chunk_size=1000,
    chunk_overlap=200,
    keep_tables_intact=True,
)

print(f"Total chunks: {len(chunks)}")
print(f"\nChunk size distribution:")
sizes = [len(c.content) for c in chunks]
print(f"  Min: {min(sizes)}, Max: {max(sizes)}, Avg: {sum(sizes)//len(sizes)}")

print(f"\n--- Sample Chunk (index 5) ---")
if len(chunks) > 5:
    c = chunks[5]
    print(f"Type: {c.type} | Page: {c.page} | Section: {c.section_title}")
    print(f"Content ({len(c.content)} chars):")
    print(c.content[:300])

Total chunks: 343

Chunk size distribution:
  Min: 1, Max: 9212, Avg: 305

--- Sample Chunk (index 5) ---
Type: text | Page: 1 | Section: FORM 10-Q
Content (48 chars):
For the quarterly period ended December 27, 2025


## 6. Parse All PDFs

In [9]:
# Parse all PDFs in the pdf folder
all_results = {}

for pdf_path in pdf_files:
    print(f"Parsing {pdf_path.name}...", end=" ")
    elements = parse_pdf(pdf_path)
    all_results[pdf_path.name] = elements
    print(f"{len(elements)} elements")

print(f"\nTotal: {sum(len(v) for v in all_results.values())} elements from {len(all_results)} PDFs")

[INFO] 2026-05-08 17:55:30,911 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-08 17:55:30,920 [RapidOCR] download_file.py:60: File exists and is valid: /opt/python-3.11-dev/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-08 17:55:30,921 [RapidOCR] main.py:57: Using /opt/python-3.11-dev/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-08 17:55:30,986 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-08 17:55:30,987 [RapidOCR] download_file.py:60: File exists and is valid: /opt/python-3.11-dev/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-08 17:55:30,988 [RapidOCR] main.py:57: Using /opt/python-3.11-dev/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-08 17:55:31,015 [RapidOCR] base.py:22: Using engine_name: onnxruntime


Parsing 10Q-Q1-2026-as-filed.pdf... 

[INFO] 2026-05-08 17:55:31,030 [RapidOCR] download_file.py:60: File exists and is valid: /opt/python-3.11-dev/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_rec_mobile.onnx
[INFO] 2026-05-08 17:55:31,030 [RapidOCR] main.py:57: Using /opt/python-3.11-dev/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_rec_mobile.onnx
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown()

333 elements
Parsing 10Q-Q2-2026-as-filed.pdf... 

Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument i

370 elements
Parsing GOOG-10-Q-Q1-2026.pdf... 

Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument i

632 elements
Parsing form-10-q.pdf... 

Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument i

1186 elements

Total: 2521 elements from 4 PDFs


In [10]:
# Summary table
print(f"{'File':<35} {'Elements':>8} {'Text':>6} {'Tables':>7} {'Headings':>8}")
print("-" * 70)
for name, elements in all_results.items():
    types = Counter(e.type for e in elements)
    print(
        f"{name:<35} {len(elements):>8} "
        f"{types.get('text', 0):>6} "
        f"{types.get('table', 0):>7} "
        f"{types.get('heading', 0):>8}"
    )

File                                Elements   Text  Tables Headings
----------------------------------------------------------------------
10Q-Q1-2026-as-filed.pdf                 333    207      24      102
10Q-Q2-2026-as-filed.pdf                 370    233      27      110
GOOG-10-Q-Q1-2026.pdf                    632    408      62      162
form-10-q.pdf                           1186    849      86      251
